# Bullinger 整页图片识别 · 上传即可试用

**整页上传 → 轮廓切行 → 公开 Bullinger TrOCR → mBERT + 历史词表 → 现代地名候选。**

1. 在 Colab 的「运行时 → 更改运行时类型」中选择 **T4 GPU**（CPU 也能运行，但慢）。
2. 从上到下运行。第 2 步会出现上传按钮，每次选择一张 JPG、PNG、WebP 或单页 TIFF。
3. 查看切行预览、逐行图片与转录、地名候选表。最后下载 JSON 或完整结果 ZIP。

初次需要下载模型与运行环境，可能需要数分钟。无需挂载 Drive 或登录 Hugging Face。
同一会话换图时，从第 2 步重新运行即可。每次创建新结果目录，保留之前的结果。

本笔记本固定使用已验证的 [GitHub 代码版本 fb7a912](https://github.com/sssstong55t-sudo/Bullinger_htr/tree/fb7a912160c0ab51fae8bf20efb23287e65b96e8)。
适用于 Bullinger 德语/拉丁语手稿；新图的效果需要人工查看。没有人工标准转录时，不计算 CER/WER。
多个现代地名候选全部保留，单个候选也仍需确认。缺少名称或坐标时保持 null。


## 1. 安装环境并下载模型（每个会话只需一次）
使用独立 Python 环境，避免改变 Colab 已加载的库。HTR 在可用时使用 GPU，mBERT 使用 CPU。


In [ ]:
import os, sys, json, subprocess, hashlib, shutil, zipfile
from pathlib import Path
from urllib.request import urlopen, Request

COMMIT = 'fb7a912160c0ab51fae8bf20efb23287e65b96e8'
BASE = Path('/content/bullinger_upload_demo')
BASE.mkdir(parents=True, exist_ok=True)
REPO = BASE / ('repo_' + COMMIT[:7])
PY = BASE / 'venv/bin/python'

def run(args, cwd=None, env=None):
    subprocess.run([str(x) for x in args], cwd=cwd, env=env, check=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if not REPO.exists():
    run(['git', 'clone', 'https://github.com/sssstong55t-sudo/Bullinger_htr.git', REPO])
run(['git', '-C', REPO, 'checkout', '--detach', COMMIT])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == COMMIT

if not PY.exists():
    run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv'])
    run([sys.executable, '-m', 'virtualenv', BASE / 'venv'])
installed = BASE / 'environment_v1.ready'
if not installed.exists():
    run([PY, '-m', 'pip', 'install', '--upgrade', 'pip'])
    # CUDA 12.4 wheel also runs on CPU. No optional GLiNER download is needed.
    run([PY, '-m', 'pip', 'install', 'torch==2.6.0',
         '--index-url', 'https://download.pytorch.org/whl/cu124'])
    run([PY, '-m', 'pip', 'install', 'transformers==4.51.0',
         'huggingface-hub==0.36.2', 'Pillow>=10,<13', 'numpy>=1.24,<3',
         'scipy>=1.11,<2', 'sentencepiece==0.2.2', 'protobuf>=4,<7'])
    installed.touch()
run([PY, '-c', "import torch; from transformers import ViTImageProcessor, VisionEncoderDecoderModel, AutoModelForTokenClassification; print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU（转录较慢）')"])

os.environ['HF_HUB_DISABLE_XET'] = '1'
run([PY, REPO / 'scripts/download_public_trocr.py'], cwd=REPO)
archive = REPO / 'models/bullinger_mbert_20261022.zip'
expected = '92cd0da3f00234a9045831ed5ca70c6c7b43d8b310fc681b1df104ed12a88683'
url = 'https://github.com/sssstong55t-sudo/Bullinger_htr/releases/download/v1.0.0-mbert-20261022/bullinger_mbert_20261022.zip'
if not archive.exists() or sha256(archive) != expected:
    temp = archive.with_suffix('.part')
    print('下载 mBERT 权重……')
    with urlopen(Request(url, headers={'User-Agent': 'Bullinger-Colab'}), timeout=120) as response, temp.open('wb') as out:
        shutil.copyfileobj(response, out, length=8 * 1024 * 1024)
    if sha256(temp) != expected:
        temp.unlink()
        raise RuntimeError('mBERT 下载校验失败，请重新运行此单元格。')
    temp.replace(archive)
with zipfile.ZipFile(archive) as z:
    destination = (REPO / 'models').resolve()
    for entry in z.infolist():
        if not (destination / entry.filename).resolve().is_relative_to(destination):
            raise ValueError('权重包包含异常路径')
    z.extractall(destination)
assert sha256(REPO / 'models/bullinger_mbert_20261022/model.safetensors') == '055be2e5c923a4e6a843e120f73cffb66eaa3cdf65a686faf6d776dfeb99ae20'
INFERENCE_ENV = {**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1', 'PYTHONUNBUFFERED': '1'}
print('环境和权重已准备好。请运行第 2 步上传图片。')


## 2. 上传一张整页图片
上传成功后显示原图。请先确认方向正确、文字清晰。PDF 请先导出为图片，多页 TIFF 请先拆页。


In [ ]:
from google.colab import files
from IPython.display import display, HTML
from PIL import Image, ImageOps
import io, uuid
from datetime import datetime, timezone

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('请每次只上传一张整页图片，然后重新运行此单元格。')
original_name, image_bytes = next(iter(uploaded.items()))
with Image.open(io.BytesIO(image_bytes)) as source:
    if getattr(source, 'n_frames', 1) != 1:
        raise ValueError('请先将多页/动画图片拆成单页。')
    oriented = ImageOps.exif_transpose(source).convert('RGBA')
    page = Image.alpha_composite(Image.new('RGBA', oriented.size, 'white'), oriented).convert('RGB')
RUN = BASE / 'runs' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
(RUN / 'input').mkdir(parents=True)
PAGE = RUN / 'input/page.png'
page.save(PAGE)
upload_info = {'original_filename': original_name,
               'original_sha256': hashlib.sha256(image_bytes).hexdigest()}
(RUN / 'upload.json').write_text(json.dumps(upload_info, ensure_ascii=False, indent=2), encoding='utf-8')
preview = page.copy()
preview.thumbnail((850, 1100))
print(f'图片：{original_name}，尺寸：{page.width} × {page.height}')
display(preview)


## 3. 整页切行并查看预览
红线是每行的轮廓，蓝色编号对应后面的转录。若页边批注、两栏或粘连文字切得不好，识别结果也会受影响。


In [ ]:
# Re-running this step creates fresh output paths for all subsequent steps.
ATTEMPT = RUN / ('attempt_' + uuid.uuid4().hex[:8])
ATTEMPT.mkdir()
CROPS = ATTEMPT / 'crops'
run([PY, REPO / 'scripts/page_segmentation_loose.py', '--image', PAGE,
     '--output-dir', CROPS], cwd=REPO, env=INFERENCE_ENV)
manifest = json.loads((CROPS / 'manifest.json').read_text())
print(f"检测到 {len(manifest['lines'])} 行。")
with Image.open(CROPS / 'preview.png') as im:
    preview = im.copy()
preview.thumbnail((1100, 1500))
display(preview)
if not manifest['lines']:
    raise ValueError('没有找到文字行，请换一张清晰、方向正确的图片。')


## 4. 转录并找地名
每处理一行会显示进度。使用公开 TrOCR 的八束解码，再将实际识别文字交给 mBERT 和历史词表。
重跑此单元格会生成新的识别文件。


In [ ]:
RESULT = ATTEMPT / ('recognition_' + uuid.uuid4().hex[:8])
RESULT.mkdir()
HTR = RESULT / 'htr.jsonl'
PLACES = RESULT / 'places.jsonl'
crop_paths = [CROPS / line['image'] for line in manifest['lines']]
run([PY, REPO / 'scripts/transcribe_images.py', '--images', *crop_paths,
     '--output', HTR, '--device', 'auto'], cwd=REPO, env=INFERENCE_ENV)
run([PY, REPO / 'scripts/extract_places.py', '--input', HTR,
     '--output', PLACES], cwd=REPO, env=INFERENCE_ENV)
print('转录和地名提取完成，继续运行第 5 步查看结果。')


## 5. 查看效果并生成网页可用的 JSON
黄色标出地名。候选表包含名称、Bullinger 地点 ID、经纬度和来源。
“单个候选待确认”表示词表给出一个候选，“多个候选”表示尚有歧义，“未匹配”表示词表中查不到。
候选坐标不代表已经完成历史地名消歧。


In [ ]:
from collections import Counter
import base64, html

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def index_by_image(rows):
    indexed = {Path(row['image']).resolve(): row for row in rows}
    assert len(indexed) == len(rows), '存在重复的行图片'
    return indexed

htr = index_by_image(read_jsonl(HTR))
ner = index_by_image(read_jsonl(PLACES))
assert set(htr) == set(ner) == set(path.resolve() for path in crop_paths), '各阶段的行图片未对齐'
counts = Counter()
lines = []
for segment in manifest['lines']:
    crop = (CROPS / segment['image']).resolve()
    prediction, entities = htr[crop], ner[crop]
    assert prediction['text'] == entities['text']
    assert prediction['image_sha256'] == sha256(crop)
    for mention in entities['places']:
        start, end = mention['start'], mention['end']
        assert 0 <= start < end <= len(prediction['text'])
        assert prediction['text'][start:end] == mention['text']
        candidates = mention['official_place_candidates']
        assert len({c['official_place_id'] for c in candidates}) == len(candidates)
        status = 'unmatched' if not candidates else ('candidate_for_review' if len(candidates) == 1 else 'ambiguous')
        assert mention['place_link_status'] == status
        counts[status] += 1
    lines.append({'line_id': segment['line_id'], 'image': 'crops/' + segment['image'],
                  'bbox': segment['bbox'], 'polygon': segment['polygon'],
                  'text': prediction['text'], 'places': entities['places']})
payload = {
    'page': {'page_id': RUN.name, **upload_info, 'image': 'input/page.png',
             'image_sha256': sha256(PAGE), 'width': manifest['page_size'][0],
             'height': manifest['page_size'][1], 'coordinate_system': manifest['coordinate_system'],
             'reading_order': manifest['reading_order']},
    'pipeline': {'github_commit': COMMIT, 'segmentation': 'contours',
                 'htr_model': 'pstroe/bullinger-general-model',
                 'htr_revision': 'be32c883a4deb812ab6f461ae79f583c3f9edc53',
                 'place_extraction_mode': 'mbert_with_aliases',
                 'place_resolution': 'official lexicon candidates for review',
                 'text_offset_unit': 'Unicode code points; end exclusive'},
    'summary': {'line_count': len(lines), 'place_mention_count': sum(counts.values()),
                'place_link_status_counts': {key: counts[key] for key in
                    ['candidate_for_review', 'ambiguous', 'unmatched']},
                'mentions_with_candidate_coordinates': sum(
                    any(c['latitude'] is not None and c['longitude'] is not None
                        for c in m['official_place_candidates']) for line in lines for m in line['places'])},
    'lines': lines}
JSON_PATH = RESULT / 'page_response.json'
JSON_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(RESULT / 'transcription.txt').write_text('\n'.join(line['text'] for line in lines) + '\n', encoding='utf-8')

def esc(value):
    return html.escape('—' if value is None else str(value))

def marked_text(text, mentions):
    # Union overlapping spans so the display preserves every original character.
    spans = []
    for mention in sorted(mentions, key=lambda m: (m['start'], m['end'])):
        start, end = mention['start'], mention['end']
        if spans and start <= spans[-1][1]:
            spans[-1][1] = max(spans[-1][1], end)
        else:
            spans.append([start, end])
    chunks, cursor = [], 0
    for start, end in spans:
        chunks += [esc(text[cursor:start]), '<mark>' + esc(text[start:end]) + '</mark>']
        cursor = end
    return ''.join(chunks) + esc(text[cursor:])

labels = {'candidate_for_review': '单个候选待确认', 'ambiguous': '多个候选', 'unmatched': '未匹配'}
body = [f"<h1>Bullinger 整页识别结果</h1><p>{esc(original_name)} · {len(lines)} 行 · {sum(counts.values())} 个地名提及</p>",
        '<p>黄色为识别出的地名。现代地名均为词表候选，尚待确认。</p>']
for line in lines:
    with Image.open(CROPS / Path(line['image']).name) as im:
        thumb = im.copy()
    thumb.thumbnail((1000, 240))
    buffer = io.BytesIO()
    thumb.save(buffer, format='PNG')
    data = base64.b64encode(buffer.getvalue()).decode('ascii')
    body.append(f"<section><h3>{esc(line['line_id'])}</h3><div class='pair'><img src='data:image/png;base64,{data}'><p>{marked_text(line['text'], line['places'])}</p></div></section>")
body.append('<h2>现代地名候选</h2>')
if not sum(counts.values()):
    body.append('<p>本次没有检测到地名，JSON 中各行 places 为 []。</p>')
else:
    body.append('<div class="scroll"><table><tr><th>行</th><th>原文地名</th><th>状态</th><th>候选名称</th><th>地点 ID</th><th>纬度</th><th>经度</th><th>来源</th></tr>')
    for line in lines:
        for mention in line['places']:
            for candidate in mention['official_place_candidates'] or [{}]:
                values = [line['line_id'], mention['text'], labels[mention['place_link_status']],
                          candidate.get('canonical_name'), candidate.get('official_place_id'),
                          candidate.get('latitude'), candidate.get('longitude'), candidate.get('source')]
                body.append('<tr>' + ''.join('<td>' + esc(v) + '</td>' for v in values) + '</tr>')
    body.append('</table></div>')
style = 'body{font-family:system-ui,sans-serif;margin:24px;color:#222}section{border-top:1px solid #ddd;padding:12px 0}.pair{display:flex;gap:20px;align-items:center;flex-wrap:wrap}.pair img{max-width:100%;width:500px}.pair p{flex:1;min-width:240px;white-space:pre-wrap;font-size:18px}mark{background:#ffe58a}table{border-collapse:collapse}td,th{border:1px solid #ddd;padding:8px;text-align:left}.scroll{overflow:auto}'
report = '<!doctype html><html lang="zh"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Bullinger 识别结果</title><style>' + style + '</style></head><body>' + ''.join(body) + '</body></html>'
(RESULT / 'report.html').write_text(report, encoding='utf-8')
display(HTML('<style>' + style + '</style>' + ''.join(body)))
print('JSON 已生成：', JSON_PATH.name)


## 6. 下载
JSON 可以交给网页使用。`start/end` 是转录文字中的 Unicode 字符位置，右端不包含。
JavaScript 可用 `Array.from(line.text).slice(start, end).join('')` 取得原文。
JSON 内的图片路径相对于 ZIP 根目录。ZIP 同时包含整页图片、切行、预览、转录、原始 JSONL 和可离线打开的 `report.html`。
Colab 会话结束前请下载需要保留的结果。


In [ ]:
# 点击本单元格左边的运行按钮，下载网页用 JSON。
files.download(str(JSON_PATH))


In [ ]:
# 可选：下载全部结果。解压后用浏览器打开 report.html 即可回看。
ZIP_PATH = RESULT / 'bullinger_page_results.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PAGE, 'input/page.png')
    for path in sorted(CROPS.iterdir()):
        if path.is_file():
            z.write(path, 'crops/' + path.name)
    for name in ['page_response.json', 'transcription.txt', 'report.html', 'htr.jsonl', 'places.jsonl']:
        z.write(RESULT / name, name)
files.download(str(ZIP_PATH))
